# Bitcoin Trading RL Quickstart

This notebook demonstrates how to use the Bitcoin Trading RL system for cryptocurrency trading.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer
from src.models.base_model import BaseModel
from src.training.trainer import Trainer
from src.evaluation.evaluator import Evaluator

## 1. Download Historical Data

Let's start by downloading some historical Bitcoin data from Binance.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(
    symbol="BTCUSDT",
    interval="5m",
    start_date="2023-01-01",
    end_date="2023-12-31"
)

# Download data
await fetcher.fetch_data()

## 2. Feature Engineering

Now let's process the raw data and create features for our model.

In [ ]:
# Initialize feature engineer
engineer = FeatureEngineer(
    input_file="data/raw/BTCUSDT_5m_data.parquet",
    output_dir="data/processed"
)

# Generate features
engineer.generate_features()

## 3. Load and Prepare Data

Let's load our processed data and prepare it for training.

In [ ]:
# Load processed data
data = pd.read_parquet("data/processed/features.parquet")

# Split into train/val/test sets
train_size = int(len(data) * 0.7)
val_size = int(len(data) * 0.15)

train_data = data[:train_size]
val_data = data[train_size:train_size+val_size]
test_data = data[train_size+val_size:]

print(f"Training data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape: {test_data.shape}")

## 4. Initialize and Train Model

Now let's create and train our trading model.

In [ ]:
# Initialize model
model = BaseModel(
    state_dim=train_data.shape[1],
    action_dim=1,
    hidden_dim=256,
    num_heads=8
)

# Create trainer
trainer = Trainer(
    model=model,
    train_data=train_data,
    val_data=val_data,
    config={
        'initial_balance': 100000,
        'transaction_fee': 0.001,
        'n_envs': 8
    }
)

# Train model
trainer.train(
    num_episodes=1000,
    validate_every=10,
    save_every=100
)

## 5. Evaluate Model

Finally, let's evaluate our trained model's performance.

In [ ]:
# Initialize evaluator
evaluator = Evaluator(
    model=model,
    test_data=test_data,
    config={
        'initial_balance': 100000,
        'transaction_fee': 0.001
    }
)

# Run backtesting
metrics = evaluator.backtest()

# Display metrics
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

## 6. Visualize Results

Let's create some visualizations of our model's performance.

In [ ]:
# Plot portfolio value over time
plt.figure(figsize=(15, 6))
plt.plot(evaluator.portfolio_values)
plt.title('Portfolio Value Over Time')
plt.xlabel('Time Steps')
plt.ylabel('Portfolio Value')
plt.grid(True)
plt.show()

# Plot position sizes
plt.figure(figsize=(15, 6))
plt.plot(evaluator.positions)
plt.title('Position Sizes Over Time')
plt.xlabel('Time Steps')
plt.ylabel('Position Size')
plt.grid(True)
plt.show()